In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer_probe_epoch50_patience10 import SwinTransformer3D, VSTFCCCLinearProbe
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-11 20:29:24,453]::3827333478::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:
backbone_checkpoint_path = './save_swintransformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_swintransformer_folder/vst_fccc_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
print(1)


1


In [6]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [7]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [8]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [9]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [10]:
num_workerssz = 20
lr = 3e-05
epochs = 50
print(1)

1


In [11]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0], fullshot=train_set_data[1], tagdata=train_set_data[2], transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0], fullshot=validation_set_data[1], tagdata=validation_set_data[2], transform=transform)
print(1)


1


In [12]:
criterion = torch.nn.L1Loss().to(device)
print(1)


1


In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mae = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(0):

    for seed_idx, seed in enumerate(SEEDS, start=1):

        logger.info(
            '================ Probe Seed {}/{}: {} ================'.format(
                seed_idx, len(SEEDS), seed
            )
        )

        set_seed(seed)

        train_generator = torch.Generator()
        train_generator.manual_seed(seed)

        val_generator = torch.Generator()
        val_generator.manual_seed(seed + 10000)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=train_generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=val_generator
        )

        # ----------------------------------------------------
        # Frozen trained backbone + freshly initialized probes
        # ----------------------------------------------------
        backbone = SwinTransformer3D()
        backbone = load_backbone_checkpoint(
            backbone,
            backbone_checkpoint_path
        )

        model = VSTFCCCLinearProbe(
            backbone=backbone,
            num_outputs=5
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.probe_heads.parameters(),
            lr=lr
        )

        probe_names = list(model.probe_heads.keys())

        seed_best = {
            name: {
                'mae': float('inf'),
                'epoch': 0
            }
            for name in probe_names
        }

        seed_best_mean_mae = float('inf')
        no_improve_count = 0

        # ----------------------------------------------------
        # Sanity checks
        # ----------------------------------------------------
        logger.info(
            'Backbone trainable params: {}'.format(
                sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
            )
        )
        logger.info(
            'Probe trainable params: {}'.format(
                sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad)
            )
        )
        logger.info('Probe dimensions: {}'.format(model.probe_dims))

        for epoch in range(epochs):

            # ==================================================
            # Train probes only
            # ==================================================
            model.train()

            train_abs_sum = {name: 0.0 for name in probe_names}
            train_count = {name: 0 for name in probe_names}

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc='Probe Seed {} Train Epoch {}'.format(seed, epoch + 1)
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                outputs = model(
                    fullshot,
                    audiodata
                )

                losses = {
                    name: criterion(outputs[name], big_five_data)
                    for name in probe_names
                }

                loss = torch.stack(
                    [losses[name] for name in probe_names]
                ).mean()

                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    for name in probe_names:
                        train_abs_sum[name] += (
                            outputs[name] - big_five_data
                        ).abs().sum().item()
                        train_count[name] += big_five_data.numel()

            train_mae = {
                name: train_abs_sum[name] / train_count[name]
                for name in probe_names
            }

            # ==================================================
            # Validation
            # ==================================================
            model.eval()

            val_abs_sum = {name: 0.0 for name in probe_names}
            val_count = {name: 0 for name in probe_names}

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc='Probe Seed {} Val Epoch {}'.format(seed, epoch + 1)
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    outputs = model(
                        fullshot,
                        audiodata
                    )

                    for name in probe_names:
                        val_abs_sum[name] += (
                            outputs[name] - big_five_data
                        ).abs().sum().item()
                        val_count[name] += big_five_data.numel()

            val_mae = {
                name: val_abs_sum[name] / val_count[name]
                for name in probe_names
            }

            mean_val_mae = float(
                np.mean([val_mae[name] for name in probe_names])
            )

            # Each representation keeps its own best epoch/result.
            for name in probe_names:
                if val_mae[name] < seed_best[name]['mae']:
                    seed_best[name]['mae'] = float(val_mae[name])
                    seed_best[name]['epoch'] = epoch + 1

            # Patience is based only on the mean validation MAE across probes.
            if mean_val_mae < seed_best_mean_mae:
                seed_best_mean_mae = mean_val_mae
                no_improve_count = 0

                if mean_val_mae < global_best_mean_mae:
                    global_best_mean_mae = mean_val_mae
                    global_best_seed = seed
                    global_best_epoch = epoch + 1

                    torch.save(
                        {
                            'seed': seed,
                            'epoch': epoch + 1,
                            'probe_model': model.probe_heads.state_dict(),
                            'mean_val_mae': mean_val_mae,
                            'per_probe_val_mae': val_mae,
                            'probe_dims': model.probe_dims
                        },
                        probe_best_model_path
                    )
            else:
                no_improve_count += 1

            print(
                '\nSeed {} | Epoch {:03d} | Mean Val MAE={:.6f} | Patience={}/{}'.format(
                    seed,
                    epoch + 1,
                    mean_val_mae,
                    no_improve_count,
                    patience
                )
            )
            print(
                '{:<22} {:>12} {:>12} {:>12}'.format(
                    'Representation',
                    'Train MAE',
                    'Val MAE',
                    '1-MAE'
                )
            )
            print('-' * 62)

            for name in probe_names:
                print(
                    '{:<22} {:>12.6f} {:>12.6f} {:>12.6f}'.format(
                        name,
                        train_mae[name],
                        val_mae[name],
                        1.0 - val_mae[name]
                    )
                )

            if no_improve_count >= patience:
                logger.info(
                    'Probe early stopping: seed={} at epoch {}. '
                    'Best mean val MAE={:.6f}'.format(
                        seed,
                        epoch + 1,
                        seed_best_mean_mae
                    )
                )
                break

            torch.cuda.empty_cache()

        seed_results.append(
            {
                'seed': seed,
                'representations': {
                    name: {
                        'best_epoch': seed_best[name]['epoch'],
                        'best_mae': seed_best[name]['mae'],
                        'best_1_mae': 1.0 - seed_best[name]['mae']
                    }
                    for name in probe_names
                }
            }
        )

        del model
        del backbone
        del optimizer
        del train_dataloader
        del val_dataloader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


# ============================================================
# Final multi-seed probe statistics
# ============================================================
print('\n' + '=' * 94)
print('FINAL VST-FCCC LAYERWISE LINEAR PROBE RESULTS')
print('=' * 94)
print(
    '{:<22} {:>14} {:>14} {:>14} {:>14}'.format(
        'Representation',
        'MAE Mean',
        'MAE Std',
        '1-MAE Mean',
        '1-MAE Std'
    )
)
print('-' * 94)

for name in probe_names:
    maes = np.array(
        [
            result['representations'][name]['best_mae']
            for result in seed_results
        ],
        dtype=np.float64
    )

    one_maes = 1.0 - maes

    print(
        '{:<22} {:>14.6f} {:>14.6f} {:>14.6f} {:>14.6f}'.format(
            name,
            maes.mean(),
            maes.std(ddof=1),
            one_maes.mean(),
            one_maes.std(ddof=1)
        )
    )

print('-' * 94)
print(
    'Global best mean-probe checkpoint: seed={}, epoch={}, mean MAE={:.6f}'.format(
        global_best_seed,
        global_best_epoch,
        global_best_mean_mae
    )
)
print('Saved probe checkpoint: {}'.format(probe_best_model_path))
print('=' * 94)


[2026-09-11 20:29:32,160]::2529951574::INFO::================ Probe Seed 1/1: 42 ================


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


[2026-09-11 20:29:34,704]::2529951574::INFO::Backbone trainable params: 0
[2026-09-11 20:29:34,705]::2529951574::INFO::Probe trainable params: 94175
[2026-09-11 20:29:34,706]::2529951574::INFO::Probe dimensions: {'stage1_block1': 96, 'stage1_block2': 96, 'stage1': 192, 'stage2_block1': 192, 'stage2_block2': 192, 'stage2': 384, 'stage3_block1': 384, 'stage3_block2': 384, 'stage3_block3': 384, 'stage3_block4': 384, 'stage3_block5': 384, 'stage3_block6': 384, 'stage3': 768, 'stage4_block1': 768, 'stage4_block2': 768, 'stage4': 1536, 'final_video': 1536, 'final_cross': 1536, 'final_fused': 3072}
Probe Seed 42 Train Epoch 1: 100%|█████████▉| 1497/1498 [02:00<00:00, 13.34it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(inp


Seed 42 | Epoch 001 | Mean Val MAE=0.112050 | Patience=0/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.289758     0.158276     0.841724
stage1_block2              0.244529     0.161028     0.838972
stage1                     0.230451     0.166092     0.833908
stage2_block1              0.212996     0.145888     0.854112
stage2_block2              0.188429     0.145963     0.854037
stage2                     0.163222     0.124213     0.875787
stage3_block1              0.119362     0.105439     0.894561
stage3_block2              0.111541     0.102911     0.897089
stage3_block3              0.116775     0.100379     0.899621
stage3_block4              0.102600     0.098911     0.901089
stage3_block5              0.100516     0.096412     0.903588
stage3_block6              0.143969     0.100791     0.899209
stage3                     0.083108     0.094220     0.905780
stage4_b

Probe Seed 42 Val Epoch 2: 100%|██████████| 498/498 [00:46<00:00, 10.67it/s]


...

Probe Seed 42 Val Epoch 49: 100%|██████████| 498/498 [00:47<00:00, 10.55it/s]



Seed 42 | Epoch 049 | Mean Val MAE=0.089176 | Patience=0/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.084428     0.101379     0.898621
stage1_block2              0.083765     0.100996     0.899004
stage1                     0.062171     0.091701     0.908299
stage2_block1              0.061984     0.091718     0.908282
stage2_block2              0.060694     0.091790     0.908210
stage2                     0.055285     0.087789     0.912211
stage3_block1              0.054154     0.087887     0.912113
stage3_block2              0.054090     0.087636     0.912364
stage3_block3              0.054055     0.086975     0.913025
stage3_block4              0.053994     0.086719     0.913281
stage3_block5              0.053864     0.086497     0.913503
stage3_block6              0.053837     0.086587     0.913413
stage3                     0.055146     0.088162     0.911838
stage4_b

Probe Seed 42 Val Epoch 50: 100%|██████████| 498/498 [00:46<00:00, 10.66it/s]


Seed 42 | Epoch 050 | Mean Val MAE=0.088609 | Patience=0/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.084347     0.101156     0.898844
stage1_block2              0.083627     0.100813     0.899187
stage1                     0.062029     0.091361     0.908639
stage2_block1              0.061832     0.091343     0.908657
stage2_block2              0.060566     0.091285     0.908715
stage2                     0.054961     0.087911     0.912089
stage3_block1              0.053943     0.086800     0.913200
stage3_block2              0.053897     0.086921     0.913079
stage3_block3              0.053814     0.086714     0.913286
stage3_block4              0.053761     0.086512     0.913488
stage3_block5              0.053608     0.086246     0.913754
stage3_block6              0.053627     0.086496     0.913504
stage3                     0.054997     0.086904     0.913096
stage4_b


/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
